In [24]:
import mlflow
from mlflow.client import MlflowClient
from mlflow.entities import ViewType
import os
import sys
import h2o
import pickle
import lightgbm as lgb
from catboost import CatBoostRegressor
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from atow_estimation.paths import PROCESSED_DATA_DIR, MODELS_DIR
from atow_estimation.config import (
    M1_M_FEATURES,
    M2_M_FEATURES,
    H_FEATURES,
    M_FEATURES,
)
from atow_estimation.utils import (
    preprocess_lightgbm,
    preprocess_xgboost,
    preprocess_catboost,
    preprocess_h2o,
    get_model_metrics
)
from atow_estimation.modeling.train import load_datasets, open_model

In [25]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "jaguevara"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "jaguevara_mlflow"
os.environ["MLFLOW_TRACKING_URI"] = "http://192.168.25.110:5000"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "HEAVY_atow_estimation"
mlflow.set_tracking_uri(uri="http://192.168.25.110:5000")
mlflow.set_experiment("HEAVY_atow_estimation")
h2o.init(nthreads=-1)

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,20 hours 45 mins
H2O_cluster_timezone:,Europe/Madrid
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,3 months and 3 days
H2O_cluster_name:,H2O_from_python_jaguevara_z8lq2e
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,929 Mb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


## Filter out runs with no feature selection

In [26]:
client = MlflowClient()
filter_string = "tags.feature_selection = '1'"
try:
    finished_runs = client.search_runs(
        experiment_ids=[55], 
        filter_string="tags.feature_selection = '1'",
        order_by=["attributes.end_time DESC"],
        # max_results=30,
        run_view_type=ViewType.ACTIVE_ONLY 
    )
    # print(f"Found {len(finished_runs)} finished runs:")
    run_ids = []
    for run in finished_runs:
        # print(run.info.run_id)
        run_ids.append(run.info.run_id)

except mlflow.exceptions.MlflowException as e:
    print(f"An MLflow error occurred: {e}")
    print("Please ensure your MLflow Tracking Server is running and accessible.")
    print("You can set MLFLOW_TRACKING_URI environment variable or use mlflow.set_tracking_uri().")

## Retrieve run id, model binary file (pkl), dataset name

In [27]:
dataset_dict = {'All flights (H wake category)': 'H_FEATURES',
                'All flights (M wake category)': 'M_FEATURES',
                'Flights with climb phase trajectory (M wake category)': 'M1_M_FEATURES',
                'Flights without climb phase trajectory (M wake category)': 'M2_M_FEATURES'
                }

models = []
for rid in run_ids:
    client = MlflowClient()
    run = client.get_run(rid)
    dataset = run.inputs.dataset_inputs[0].dataset
    dataset_name = dataset.name
    run_name = run.data.tags.get("mlflow.runName")  
    models.append((rid, run_name, dataset_dict[dataset_name]))

## Train Test Split

In [28]:
data = load_datasets()

data_m1_M = data['data_m1_M']
data_m2_M = data['data_m2_M']
data_H = data['data_H']
data_M = data['data_M']
all_data = [data_m1_M, data_m2_M, data_H, data_M]
selected_features = [M1_M_FEATURES, M2_M_FEATURES, H_FEATURES, M_FEATURES]
selected_features_str = ['M1_M_FEATURES', 'M2_M_FEATURES', 'H_FEATURES', 'M_FEATURES']

In [29]:
train_data = {}

for d, feat, feat_str in zip(all_data, selected_features, selected_features_str):
    train_h2o, test_h2o, X_test_df, y_test_np = preprocess_h2o(d[feat])
    train_data[f'train_h2o_{feat_str}'] = train_h2o
    
for d, feat, feat_str in zip(all_data, selected_features, selected_features_str):
    X_train, X_test, y_train, y_test, *pools = preprocess_xgboost(d[feat])
    train_data[f'train_xgboost_{feat_str}'] = (X_train, y_train)
    
for d, feat, feat_str in zip(all_data, selected_features, selected_features_str):
    X_train, X_test, y_train, y_test, *pools = preprocess_lightgbm(d[feat])
    train_data[f'train_lightgbm_{feat_str}'] = (X_train, y_train)
    
for d, feat, feat_str in zip(all_data, selected_features, selected_features_str):
    X_train, X_test, y_train, y_test, *pools = preprocess_catboost(d[feat])
    train_data[f'train_catboost_{feat_str}'] = (X_train, y_train)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


## Metrics for h2o models

In [7]:
for model in models:
    if model[1].startswith('h2o'):
        m = h2o.load_model(os.path.join(MODELS_DIR, model[1]))
        for s in train_data:
            if s.endswith('_h2o_' + model[2]):
                t = train_data[s]
        
        predictions = m.predict(t).as_data_frame().values.flatten()
        y_train = t['tow'].as_data_frame().values.flatten()
        _, mae_train, rmse_train, _ = get_model_metrics(predictions, y_train)
        
        with mlflow.start_run(run_id=model[0]) as run:
            mlflow.log_metrics({"MAE_train": mae_train})
            mlflow.log_metrics({"RMSE_train": rmse_train})

stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%


c:\Users\jaguevara\AppData\Local\anaconda3\envs\TFM\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
c:\Users\jaguevara\AppData\Local\anaconda3\envs\TFM\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


🏃 View run h2o_20250612_021237 at: http://192.168.25.110:5000/#/experiments/55/runs/3a6907d3ea024dd4b501d854af6d0053
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%


c:\Users\jaguevara\AppData\Local\anaconda3\envs\TFM\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
c:\Users\jaguevara\AppData\Local\anaconda3\envs\TFM\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


🏃 View run h2o_20250612_015518 at: http://192.168.25.110:5000/#/experiments/55/runs/1d578025cb9a4792819195544682147c
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%


...

🏃 View run h2o_20250610_122016 at: http://192.168.25.110:5000/#/experiments/55/runs/c5cdfe0d18b74148a05e71c03eca5c99
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%


c:\Users\jaguevara\AppData\Local\anaconda3\envs\TFM\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
c:\Users\jaguevara\AppData\Local\anaconda3\envs\TFM\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


🏃 View run h2o_20250610_121437 at: http://192.168.25.110:5000/#/experiments/55/runs/d9f41fffb33941f0a3b8a5cb5ade535b
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%


c:\Users\jaguevara\AppData\Local\anaconda3\envs\TFM\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
c:\Users\jaguevara\AppData\Local\anaconda3\envs\TFM\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


🏃 View run h2o_20250610_120925 at: http://192.168.25.110:5000/#/experiments/55/runs/5eb2e12e71c045cd96d5b19c66ffdb48
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55


## Metrics for the rest of the models

In [7]:
lgbm_exceptions = ['lightgbm_20250611_125252', 'lightgbm_20250611_124414', 'lightgbm_20250611_123903', 'lightgbm_20250611_123311']

In [34]:
X_train['RECATwake'].unique()

array(['H+', 'H-'], dtype=object)

In [36]:
import numpy as np
import pandas as pd

In [38]:
logged_model = 'runs:/832f5be9a745432b8fcc4b2243011322/lightgbm_20250611_125252'
loaded_model = mlflow.pyfunc.load_model(logged_model)
for s in train_data:
    if s.endswith('_lightgbm_H_FEATURES'):
        print(s)
        X_train = train_data[s][0]
        y_train = train_data[s][1]

categorical_mappings = {
    "aircraftType": ['A320', 'B772', 'E190', 'A333', 'A20N', 'B738', 'A21N', 'A321',
       'A319', 'BCS3', 'B38M', 'A343', 'BCS1', 'B788', 'B789', 'A332',
       'B737', 'E195', 'B763', 'CRJ9', 'B77W', 'A359', 'B39M', 'B739',
       'B752'],
    "airlineCode": ['BEL', 'AAL', 'JAF', 'THY', 'SAS', 'TOM', 'SWR', 'EIN', 'AUA',
       'EWG', 'TUI', 'TFL', 'TRA', 'BLX', 'EDW', 'AEA', 'SZS'],
    "RECATwake": ['M+', 'H+', 'M-', 'H-'],
    "routeType": ['OVERFLIGHT', 'INTERNATIONAL', 'NATIONAL'],
}

for col, allowed in categorical_mappings.items():
    X_train[col] = pd.Categorical(X_train[col], categories=allowed)
    X_train[col] = X_train[col].astype(str)
    
        
y_pred = loaded_model.predict(X_train)
_, mae_train, rmse_train, _ = get_model_metrics(y_pred, y_train)

# m = CatBoostRegressor()
# m.load_model('../models/catboost_20250611_125647.cb')

# for s in train_data:
#     if s.endswith('_catboost_M1_M_FEATURES'):
#         X_train = train_data[s][0]
#         y_train = train_data[s][1]
# y_pred = m.predict(X_train)
# _, mae_train, rmse_train, _ = get_model_metrics(y_pred, y_train)

# with mlflow.start_run(run_id="02c3c47dafba4e7f9accd522aaeaab22") as run:
#     mlflow.log_metrics({"MAE_train": mae_train})
#     mlflow.log_metrics({"RMSE_train": rmse_train})

2025/07/01 10:33:11 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - graphviz (current: 0.20.3, required: graphviz==0.20.1)
 - matplotlib (current: 3.10.1, required: matplotlib==3.10.0)
 - numpy (current: 1.26.4, required: numpy==2.2.5)
 - pyarrow (current: 19.0.1, required: pyarrow==19.0.0)
 - scipy (current: 1.15.2, required: scipy==1.15.3)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2025/07/01 10:33:12 WARNING mlflow.pyfunc: The version of Python that the model was saved in, `Python 3.13.4`, differs from the version of Python that is currently running, `Python 3.10.17`, and may be incompatible


train_lightgbm_H_FEATURES


ValueError: train and valid dataset categorical_feature do not match.

In [8]:
for model in models:
    if model[1].startswith('lightgbm'):
        print(model[1])
        if model[1] not in lgbm_exceptions:
            with open(os.path.join(MODELS_DIR, f"{model[1]}.pkl"), "rb") as f:
                m = pickle.load(f)
        else:
            continue
        for s in train_data:
            if s.endswith('_lightgbm_' + model[2]):
                print(model[2])
                X_train = train_data[s][0]
                y_train = train_data[s][1]
        y_pred = m.predict(X_train)
        _, mae_train, rmse_train, _ = get_model_metrics(y_pred, y_train)
        
        with mlflow.start_run(run_id=model[0]) as run:
            mlflow.log_metrics({"MAE_train": mae_train})
            mlflow.log_metrics({"RMSE_train": rmse_train})

lightgbm_20250610_013955
H_FEATURES
🏃 View run lightgbm_20250610_013955 at: http://192.168.25.110:5000/#/experiments/55/runs/796d8868ad4946a5a4dd1cc2daccfbee
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
lightgbm_20250610_012438
M_FEATURES
🏃 View run lightgbm_20250610_012438 at: http://192.168.25.110:5000/#/experiments/55/runs/cb0677fa5fae43789a56ed14da1dbe73
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
lightgbm_20250610_011510
M2_M_FEATURES
🏃 View run lightgbm_20250610_011510 at: http://192.168.25.110:5000/#/experiments/55/runs/e8cebf3e4e4641e1aa753ad74e041bab
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
lightgbm_20250610_010507
M1_M_FEATURES
🏃 View run lightgbm_20250610_010507 at: http://192.168.25.110:5000/#/experiments/55/runs/3ff8ad6faaf240dfa127d04db0480216
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
lightgbm_20250609_223715
H_FEATURES
🏃 View run lightgbm_20250609_223715 at: http://192.168.25.110:

In [9]:
for model in models:
    if model[1].startswith('xgboost'):
        print(model[1])
        with open(os.path.join(MODELS_DIR, f"{model[1]}.pkl"), "rb") as f:
            m = pickle.load(f)        
        for s in train_data:
            if s.endswith('_xgboost_' + model[2]):
                print(model[2])
                X_train = train_data[s][0]
                y_train = train_data[s][1]
        y_pred = m.predict(X_train)
        _, mae_train, rmse_train, _ = get_model_metrics(y_pred, y_train)
        
        with mlflow.start_run(run_id=model[0]) as run:
            mlflow.log_metrics({"MAE_train": mae_train})
            mlflow.log_metrics({"RMSE_train": rmse_train})

xgboost_20250610_105633
M1_M_FEATURES
🏃 View run xgboost_20250610_105633 at: http://192.168.25.110:5000/#/experiments/55/runs/7d37d7da703743e5a4938edaefdf3f18
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
xgboost_20250610_110832
M2_M_FEATURES
🏃 View run xgboost_20250610_110832 at: http://192.168.25.110:5000/#/experiments/55/runs/429e954a90e44542889f308ff5270f3f
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
xgboost_20250610_111216
M_FEATURES
🏃 View run xgboost_20250610_111216 at: http://192.168.25.110:5000/#/experiments/55/runs/eac67facc2eb4e2babd3d01b9d3bc774
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
xgboost_20250610_111638
H_FEATURES
🏃 View run xgboost_20250610_111638 at: http://192.168.25.110:5000/#/experiments/55/runs/ff913bae73c24a16a420b095eef2c3c4
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
xgboost_20250612_005603
M1_M_FEATURES
🏃 View run xgboost_20250612_005603 at: http://192.168.25.110:5000/#/

In [12]:
m = CatBoostRegressor()
m.load_model('../models/catboost_20250611_125647.cb')

for s in train_data:
    if s.endswith('_catboost_M1_M_FEATURES'):
        X_train = train_data[s][0]
        y_train = train_data[s][1]
y_pred = m.predict(X_train)
_, mae_train, rmse_train, _ = get_model_metrics(y_pred, y_train)

with mlflow.start_run(run_id="02c3c47dafba4e7f9accd522aaeaab22") as run:
    mlflow.log_metrics({"MAE_train": mae_train})
    mlflow.log_metrics({"RMSE_train": rmse_train})

🏃 View run catboost_20250611_125647 at: http://192.168.25.110:5000/#/experiments/55/runs/02c3c47dafba4e7f9accd522aaeaab22
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55


In [12]:
for model in models:
    if model[1].startswith('catboost'):
        print(model[1])
        if model[1] != 'catboost_20250611_125647':
            with open(os.path.join(MODELS_DIR, f"{model[1]}.pkl"), "rb") as f:
                m = pickle.load(f)
        else:
            continue        
        for s in train_data:
            if s.endswith('_catboost_' + model[2]):
                print(model[2])
                X_train = train_data[s][0]
                y_train = train_data[s][1]
        y_pred = m.predict(X_train)
        _, mae_train, rmse_train, _ = get_model_metrics(y_pred, y_train)
        
        with mlflow.start_run(run_id=model[0]) as run:
            mlflow.log_metrics({"MAE_train": mae_train})
            mlflow.log_metrics({"RMSE_train": rmse_train})

catboost_20250610_014205
M1_M_FEATURES
🏃 View run catboost_20250610_014205 at: http://192.168.25.110:5000/#/experiments/55/runs/039c402c0f364bb3b365810d7a4f0c83
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
catboost_20250610_021319
M2_M_FEATURES
🏃 View run catboost_20250610_021319 at: http://192.168.25.110:5000/#/experiments/55/runs/09150dae2f794f43885b99f52ca453d1
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
catboost_20250610_103355
M_FEATURES
🏃 View run catboost_20250610_103355 at: http://192.168.25.110:5000/#/experiments/55/runs/a2b218f07e204f58b477ea8824c25aa9
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
catboost_20250610_104959
H_FEATURES
🏃 View run catboost_20250610_104959 at: http://192.168.25.110:5000/#/experiments/55/runs/8fbe771043bd4c11bb679abd2b93df50
🧪 View experiment at: http://192.168.25.110:5000/#/experiments/55
catboost_20250611_125647
catboost_20250611_214113
M2_M_FEATURES
🏃 View run catboost_20250611_21411